# Ablation and steering experiments
We're going to
- Load the data for helix face value bins on train sequences
- Load all of our test proteins
- Iterate through our test set by randomly picking residues, then doing three different steering experiments:
    1. Mask residue. On layer 18, subtract the mean activation for a residue's current angle, then add the global mean back
        - Controls: subtract mean for random activation instead of bin's own, subtract random vector of same norm
    2. Fit OLS on mean activations, subtract the OLS fitted activation for current value then add back the fitted value for the global mean
        - Controls: subtract fitted activation of random value instead of residue's own, subtract random vector of same norm
    3. Same as above but fit a + b \cos \theta + c \sin \theta instead of OLS
- Endpoints:
    -   Log likelihood of the correct aka. original aminoacid
    -   Top-1 accuracy (is the top prediction changing ?)
    -   95% confidence intervals from each, computed from deciles for effect 

In [1]:
import hashlib

import numpy as np
import polars as pl
import torch
from transformers import AutoTokenizer, EsmForMaskedLM

from biomanifolds import config
from biomanifolds.datasets import swissprot
from biomanifolds.labels import af2_structure_labels

LAYERS = [18, 28, 30]
MODEL = "facebook/esm2_t30_150M_UR50D"
N_RESIDUES = 10000
rng = np.random.default_rng(0)

# Bin sums and counts of helix face on the train set, for each layer (from notebook 2): bin k is [k - 180, k - 179) degrees
helix_sums, helix_counts = {}, {}
for layer in LAYERS:
    bin_files = {name: config.data_path("label_bins") / f"train_{name}_L{layer}.npy" for name in [
        "helix_face_1deg_bin_sums", "helix_face_1deg_bin_counts"]}
    assert all(path.exists() for path in bin_files.values()), f"no layer-{layer} bin files found, run notebook 2 first"
    helix_sums[layer], helix_counts[layer] = (np.load(path) for path in bin_files.values())

# ESM-2 with its amino-acid head, to read the log-likelihood of the masked residue
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL, attn_implementation="sdpa").eval().to(device)


# The sampled test proteins (the same 10% MD5 rule as notebook 2)
def in_sample(accession):
    return int(hashlib.md5(accession.encode()).hexdigest()[:8], 16) / 16**8 < 0.10

sequences = pl.read_parquet(swissprot.swissprot_dir() / "swissprot.parquet", columns=["accession", "sequence"])
splits = pl.read_parquet(swissprot.swissprot_dir() / "splits" / "splits.parquet", columns=["accession", "split"])
test_proteins = (
    sequences.join(splits.filter(pl.col("split") == "test"), on="accession")
    .filter(pl.col("accession").map_elements(in_sample, return_dtype=pl.Boolean))
    .select("accession", "sequence")
    .sort("accession")
)
test_proteins

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  595MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/491 [00:00<?, ?it/s]

accession,sequence
str,str
"""A0A023PZF5""","""MQNIFYRSAYVHTVCPARLEYYLLLPLLLY…"
"""A0A072VMJ3""","""MGFDNPRSERFEDDPEISKIPTTSGVKVKY…"
"""A0A075TXZ8""","""MYSTFSANFDTTIDASRDRSATPVTAPRPK…"
"""A0A077Y877""","""MHQDNSMRRAINQSRNGGSDSCDINNDREH…"
"""A0A084B9Z3""","""MSTMAKSPEANNLHQDVIAQFPILNGYTHT…"
…,…
"""V9TSX0""","""MRKWLIFLLIAAVAGLSACSTSGNTTVSDD…"
"""W0TEF9""","""MLRSTLSNWREYLTPITHTSTFETTGQLTP…"
"""W1QDU5""","""MSNSPPSSSSSGKGLSIPRKLALNAINEIQ…"


## Fitting OLS on activations, as well as activations transformed by harmonic functions

In [2]:
theta = np.deg2rad(np.arange(360) - 179.5)  # centre of each 1-degree bin, in radians
H, global_mean, B_linear, B_harmonic = {}, {}, {}, {}
for layer in LAYERS:
    H[layer] = helix_sums[layer] / helix_counts[layer][:, None]  # (360, 640) mean activation per bin
    global_mean[layer] = helix_sums[layer].sum(0) / helix_counts[layer].sum()  # mu: mean activation over all train helix residues
    weights = np.sqrt(helix_counts[layer])[:, None]  # least squares on the bin means, weighted by the residues per bin

    # OLS on linear inputs
    X = np.column_stack([np.ones(360), theta])
    B_linear[layer] = np.linalg.lstsq(X * weights, H[layer] * weights, rcond=None)[0]

    # OLS on inputs transformed by harmonic functions
    X = np.column_stack([np.ones(360), np.cos(theta), np.sin(theta)])
    B_harmonic[layer] = np.linalg.lstsq(X * weights, H[layer] * weights, rcond=None)[0]

# Mean activation for a specific value of the latent
def bin_mean_activation(t, layer):
    return H[layer][int(np.floor(np.rad2deg(t) + 180)) % 360]

def linear_activation(t, layer):
    return np.array([1.0, t]) @ B_linear[layer]

def harmonic_activation(t, layer):
    return np.array([1.0, np.cos(t), np.sin(t)]) @ B_harmonic[layer]

# Object containing the fitted activations for each angle according to each method
activation_at_angle = {"bin_means": bin_mean_activation, "linear": linear_activation, "harmonic": harmonic_activation}

# weighted R^2 of the two fits on the bin means, for each layer
r2 = []
for layer in LAYERS:
    total = (helix_counts[layer][:, None] * (H[layer] - global_mean[layer]) ** 2).sum()
    for name, activation in [("linear", linear_activation), ("harmonic", harmonic_activation)]:
        fitted = np.stack([activation(t, layer) for t in theta])
        r2.append({"layer": layer, "fit": name, "r2": 1 - (helix_counts[layer][:, None] * (H[layer] - fitted) ** 2).sum() / total})
pl.DataFrame(r2)

layer,fit,r2
i64,str,f64
18,"""linear""",0.152217
18,"""harmonic""",0.870341
28,"""linear""",0.168054
28,"""harmonic""",0.953803
30,"""linear""",0.114587
30,"""harmonic""",0.958772


## Randomly sampling 10k residues from the test dataset

In [3]:
residues = (
    af2_structure_labels.load_labels(["accession", "i", "helix_face"])
    .filter(pl.col("helix_face").is_not_null())
    .join(test_proteins.lazy(), on="accession")
    .collect(engine="streaming")
    .sort("accession", "i")  # fixed order, so that the sample is reproducible
    .sample(N_RESIDUES, seed=0)
)
residues

accession,i,helix_face,sequence
str,i32,f32,str
"""E4V6L3""",265,-136.852249,"""MAVHHKRAGSHRGRGSKAGRSRGRKFETSR…"
"""P09207""",25,-154.556427,"""MREIVHLQIGQCGNQIGAKFWEVISDEHGI…"
"""O74088""",45,137.638168,"""MVKSKVKKVEKGKEGEEKRSTYVLLKKVLI…"
"""A0L1N4""",444,-40.564983,"""MFGKLLTKVFGSRNDRTLKGLQKVVNKINA…"
"""P9WGD0""",245,109.674538,"""MPKISSRDGGRPAQRTVNPIIVTRRGKIAR…"
…,…,…,…
"""P46577""",120,71.256599,"""MAEKAENLPSSSAEASEEPSPQTGPNVNQK…"
"""Q80W92""",203,130.537308,"""MNPEKDFAPLTPNIVRALNDKLYEKRKVAA…"
"""B5F683""",77,1.851507,"""MSVPLILTLLAGAATFIGAFLGVLGQKPSN…"


## Editing activations
For each randomly sampled residue, ablate the fitted latent vector for its angle according to each different method, measure the effect on logits and the top-predicted aminoacid.

In [4]:
def run_layers(h, start, end):
    """Run ESM layers start+1 .. end on the hidden states h (batch, tokens, 640)."""
    rotary = model.esm.rotary_embeddings(h, torch.arange(h.shape[1], device=h.device)[None])  # rotary position embeddings
    for block in model.esm.encoder.layer[start:end]:
        h = block(h, position_embeddings=rotary)
    return h


results = []
for accession, i, face, sequence in residues.select("accession", "i", "helix_face", "sequence").iter_rows():
    tokens = tokenizer([sequence], return_tensors="pt").to(device)
    position = i + 1  # token 0 is <cls>
    original = tokens["input_ids"][0, position].item()
    tokens["input_ids"][0, position] = tokenizer.mask_token_id

    # the changes added to h at the masked position, for each layer: mu - f(theta)
    own, random_angle = np.deg2rad(face), rng.uniform(-np.pi, np.pi)
    deltas = {}
    for layer in LAYERS:
        deltas[layer] = {"none": np.zeros(640)}
        for method, activation in activation_at_angle.items():
            deltas[layer][method] = global_mean[layer] - activation(own, layer)
            deltas[layer][f"{method} | random angle"] = global_mean[layer] - activation(random_angle, layer)
            random_direction = rng.standard_normal(640)
            deltas[layer][f"{method} | random vector"] = random_direction / np.linalg.norm(random_direction) * np.linalg.norm(deltas[layer][method])

    # the unedited stream runs once, layer by layer; at each layer in LAYERS the edits branch off and run to the last layer
    with torch.no_grad():
        h = model.esm.embeddings(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"])
        previous = 0
        for layer in LAYERS:
            h = run_layers(h, previous, layer)  # unedited, up to `layer`
            previous = layer
            edited = h.repeat(len(deltas[layer]), 1, 1)  # one copy per edit
            edited[:, position] += torch.tensor(np.stack(list(deltas[layer].values())), dtype=torch.float32, device=device)
            edited = run_layers(edited, layer, len(model.esm.encoder.layer))
            log_probs = model.lm_head(model.esm.encoder.emb_layer_norm_after(edited))[:, position].log_softmax(-1)  # (edits, vocabulary)
            for name, lp in zip(deltas[layer], log_probs):
                results.append({"accession": accession, "i": i, "layer": layer, "edit": name,
                                "log_likelihood": lp[original].item(), "top1_correct": lp.argmax().item() == original})

results = pl.DataFrame(results)
results

accession,i,layer,edit,log_likelihood,top1_correct
str,i64,i64,str,f64,bool
"""E4V6L3""",265,18,"""none""",-0.924199,true
"""E4V6L3""",265,18,"""bin_means""",-0.957873,true
"""E4V6L3""",265,18,"""bin_means | random angle""",-0.851415,true
"""E4V6L3""",265,18,"""bin_means | random vector""",-1.057731,true
"""E4V6L3""",265,18,"""linear""",-0.995532,true
…,…,…,…,…,…
"""B8AFI3""",469,30,"""linear | random angle""",-0.957954,true
"""B8AFI3""",469,30,"""linear | random vector""",-0.938086,true
"""B8AFI3""",469,30,"""harmonic""",-1.08055,false


## Effects
Change of each endpoint against the unedited pass of the same residue, with bootstrap 95% confidence intervals of the mean.

In [5]:
from IPython.display import display

# Confidence intervals from deciles
def bootstrap_ci(x, n=10_000):
    means = rng.choice(x, (n, len(x))).mean(1)
    return np.percentile(means, [2.5, 97.5])

# Print out results with confidence intervals
unedited = results.filter(pl.col("edit") == "none").select("accession", "i", "layer", ll_unedited="log_likelihood", top1_unedited="top1_correct")
effects = results.join(unedited, on=["accession", "i", "layer"]).with_columns(
    delta_ll=pl.col("log_likelihood") - pl.col("ll_unedited"),
    delta_top1=pl.col("top1_correct").cast(pl.Float64) - pl.col("top1_unedited").cast(pl.Float64),
)
summary = []
for (layer, name), group in effects.group_by("layer", "edit", maintain_order=True):
    ll_low, ll_high = bootstrap_ci(group["delta_ll"].to_numpy())
    top1_low, top1_high = bootstrap_ci(group["delta_top1"].to_numpy())
    summary.append({"layer": layer, "edit": name, "mean_delta_ll": group["delta_ll"].mean(), "ll_ci_low": ll_low, "ll_ci_high": ll_high,
                    "top1_accuracy": group["top1_correct"].mean(), "mean_delta_top1": group["delta_top1"].mean(),
                    "top1_ci_low": top1_low, "top1_ci_high": top1_high})
summary = pl.DataFrame(summary)
pl.Config.set_tbl_rows(-1)  # show every row and column
pl.Config.set_tbl_cols(-1)
for layer in LAYERS:
    display(summary.filter(pl.col("layer") == layer))

layer,edit,mean_delta_ll,ll_ci_low,ll_ci_high,top1_accuracy,mean_delta_top1,top1_ci_low,top1_ci_high
i64,str,f64,f64,f64,f64,f64,f64,f64
18,"""none""",0.0,0.0,0.0,0.4379,0.0,0.0,0.0
18,"""bin_means""",-0.000375,-0.001088,0.00034,0.4377,-0.0002,-0.0021,0.0017
18,"""bin_means | random angle""",-0.000917,-0.001615,-0.000236,0.4366,-0.0013,-0.0032,0.0006
18,"""bin_means | random vector""",-0.001181,-0.001907,-0.00048,0.4375,-0.0004,-0.0021,0.0014
18,"""linear""",-3.3205e-7,-0.000262,0.000262,0.4376,-0.0003,-0.0014,0.0008
18,"""linear | random angle""",-0.000065,-0.000349,0.000226,0.4381,0.0002,-0.0008,0.0012
18,"""linear | random vector""",-0.000163,-0.000412,0.000087,0.4387,0.0008,-0.0002,0.0018
18,"""harmonic""",-0.000358,-0.001,0.000284,0.4383,0.0004,-0.0015,0.0023
18,"""harmonic | random angle""",-0.000949,-0.001576,-0.000307,0.437,-0.0009,-0.0026,0.0008


layer,edit,mean_delta_ll,ll_ci_low,ll_ci_high,top1_accuracy,mean_delta_top1,top1_ci_low,top1_ci_high
i64,str,f64,f64,f64,f64,f64,f64,f64
28,"""none""",0.0,0.0,0.0,0.4379,0.0,0.0,0.0
28,"""bin_means""",-0.007371,-0.009547,-0.005143,0.4376,-0.0003,-0.0033,0.0027
28,"""bin_means | random angle""",-0.005479,-0.007652,-0.003348,0.439,0.0011,-0.0019,0.0042
28,"""bin_means | random vector""",-0.002203,-0.003516,-0.00091,0.4382,0.0003,-0.0022,0.0028
28,"""linear""",-0.00049,-0.001189,0.0002,0.4382,0.0003,-0.0016,0.0022
28,"""linear | random angle""",0.000225,-0.000481,0.000926,0.4381,0.0002,-0.0015,0.0019
28,"""linear | random vector""",-0.000161,-0.000647,0.000338,0.437,-0.0009,-0.0024,0.0006
28,"""harmonic""",-0.007169,-0.00933,-0.00504,0.4378,-0.0001,-0.003,0.0028
28,"""harmonic | random angle""",-0.005346,-0.007492,-0.003194,0.4391,0.0012,-0.0019,0.0043


layer,edit,mean_delta_ll,ll_ci_low,ll_ci_high,top1_accuracy,mean_delta_top1,top1_ci_low,top1_ci_high
i64,str,f64,f64,f64,f64,f64,f64,f64
30,"""none""",0.0,0.0,0.0,0.4379,0.0,0.0,0.0
30,"""bin_means""",-0.023616,-0.027465,-0.019822,0.4345,-0.0034,-0.0073,0.0006
30,"""bin_means | random angle""",-0.0213,-0.025234,-0.017383,0.4375,-0.0004,-0.0042,0.0034
30,"""bin_means | random vector""",-0.001659,-0.002669,-0.000649,0.4376,-0.0003,-0.0028,0.0021
30,"""linear""",-0.000026,-0.000564,0.000507,0.438,0.0001,-0.0016,0.0017
30,"""linear | random angle""",-0.000129,-0.000699,0.000439,0.4382,0.0003,-0.0012,0.0018
30,"""linear | random vector""",-0.000059,-0.0004,0.000293,0.4381,0.0002,-0.0011,0.0016
30,"""harmonic""",-0.023372,-0.027155,-0.019477,0.4347,-0.0032,-0.007,0.0005
30,"""harmonic | random angle""",-0.020948,-0.024946,-0.017001,0.4381,0.0002,-0.0036,0.0039


# Results so far
- 